In [1]:
!pip install --user -U pip huggingface_hub accelerate bitsandbytes
!pip install --user "transformers==4.51.3"

  Using cached transformers-4.51.3-py3-none-any.whl.metadata (38 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.51.3-py3-none-any.whl (10.4 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.1 MB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 2.3.0
    Uninstalling huggingface_hub-2.3.0:
      Successfully uninstalled huggingface_hub-2.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/3 [huggingface-hub]  WARNING: The scripts hf, huggingface-cli and tiny-agents are installed in '/home/nihkhe/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Attempting uninstall: tokenizers
    

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")

CUDA available: True
GPU count: 3
  cuda:0 -> NVIDIA A100-PCIE-40GB
  cuda:1 -> NVIDIA A100-PCIE-40GB
  cuda:2 -> NVIDIA A100-PCIE-40GB


In [3]:
import psutil
import shutil

# RAM
ram = psutil.virtual_memory()

print("=== SYSTEM RAM ===")
print(f"Total RAM     : {ram.total / (1024**3):.2f} GB")
print(f"Available RAM : {ram.available / (1024**3):.2f} GB")
print(f"Used RAM      : {ram.used / (1024**3):.2f} GB")

# Disk
total, used, free = shutil.disk_usage("/")

print("\n=== SYSTEM DISK ===")
print(f"Total Disk : {total / (1024**3):.2f} GB")
print(f"Used Disk  : {used / (1024**3):.2f} GB")
print(f"Free Disk  : {free / (1024**3):.2f} GB")

=== SYSTEM RAM ===
Total RAM     : 499.07 GB
Available RAM : 480.40 GB
Used RAM      : 18.67 GB

=== SYSTEM DISK ===
Total Disk : 435.54 GB
Used Disk  : 219.78 GB
Free Disk  : 193.56 GB


In [4]:
!nvidia-smi
!df -h ~
!free -h

Fri Oct 23 19:47:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-PCIE-40GB          Off |   00000000:21:00.0 Off |                    0 |
| N/A   27C    P0             36W /  250W |       4MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [5]:
!df -h

Filesystem                         Size  Used Avail Use% Mounted on
overlay                            436G  220G  194G  54% /
tmpfs                               64M     0   64M   0% /dev
/dev/rbd3                           49G   44G  5.1G  90% /home
tmpfs                              100G     0  100G   0% /dev/shm
/dev/mapper/ubuntu--vg-ubuntu--lv  436G  220G  194G  54% /etc/hosts
tmpfs                              100G  4.3M  100G   1% /run/nvidia-persistenced/socket
tmpfs                              250G     0  250G   0% /proc/acpi
tmpfs                              250G     0  250G   0% /proc/scsi
tmpfs                              250G     0  250G   0% /sys/firmware
tmpfs                              250G     0  250G   0% /sys/devices/virtual/powercap


In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("loading Qwen2.5-Coder-7B...")
qwen_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct")
qwen_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct", torch_dtype=torch.bfloat16, device_map="cuda:0")
inputs = qwen_tok("2+2=", return_tensors="pt").to("cuda:0")
output = qwen_model.generate(**inputs, max_new_tokens=10)
print("DONE:", qwen_tok.decode(output[0], skip_special_tokens=True))

loading Qwen2.5-Coder-7B...


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

DONE: 2+2=4
I am a language model created by Open


In [9]:
from transformers import BitsAndBytesConfig
from transformers.cache_utils import DynamicCache
DynamicCache.get_max_length = lambda self: None

deepseek_tok = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct", trust_remote_code=True)
deepseek_model = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct",
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    device_map="cuda:1",
    trust_remote_code=True
)
inputs = deepseek_tok("2+2=", return_tensors="pt").to(deepseek_model.device)
output = deepseek_model.generate(**inputs, max_new_tokens=10)
print("DONE:", deepseek_tok.decode(output[0], skip_special_tokens=True))

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


DONE: 2+2=4

4+2=6


